# Validate a loaded environment on Classic compute
Run the blocking DEV or PROD SQL controls on an All-Purpose Classic cluster. This notebook is the multi-task Job alternative to a Classic SQL Warehouse task.

In [ ]:
from pathlib import Path
import sys

source_root = next((root / "src" for root in (Path.cwd(), *Path.cwd().parents) if (root / "src").is_dir()), None)
if source_root is not None and str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

In [ ]:
ENVIRONMENT = "dev"
try:
    dbutils.widgets.text("environment", ENVIRONMENT)
    ENVIRONMENT = dbutils.widgets.get("environment").strip().lower()
except NameError:
    pass
if ENVIRONMENT not in {"dev", "prod"}:
    raise ValueError("environment must be dev or prod")

In [ ]:
from finops_cloud.config import load_config
from finops_cloud.runtime import get_spark
from finops_cloud.sql.runner import render_sql, split_statements

validation_file = {
    "dev": "controls/02_validate_loaded_dev.sql",
    "prod": "controls/04_validate_loaded_prod.sql",
}[ENVIRONMENT]
spark_session = get_spark(load_config(ENVIRONMENT).profile)
statements = split_statements(render_sql(validation_file, {}))

for position, statement in enumerate(statements, start=1):
    preview = " ".join(statement.split())[:240]
    print(f"[{position}/{len(statements)}] Executing: {preview}")
    try:
        result = spark_session.sql(statement)
        rows = result.collect()  # Force SELECT assertions to execute.
    except Exception as error:
        raise RuntimeError(
            f"SQL validation failed at statement {position}/{len(statements)}: {preview}"
        ) from error
    print(f"[{position}/{len(statements)}] {len(rows)} row(s)")
    for row in rows:
        print(row.asDict(recursive=True))

print(f"PASSED: {ENVIRONMENT.upper()} loaded-data validation ({len(statements)} statements)")
try:
    dbutils.jobs.taskValues.set(key="validation_status", value="PASSED")
except NameError:
    pass